# KPIs

Run one full `pipeline.orchestrator.run`, then print operational counts.
Replace the schema template before running. Set `SHOW_TEXT = True` to print answers from the live objects.
The last cell deletes this collection.


In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").is_file():
    ROOT = ROOT.parent
os.chdir(ROOT)
for entry in (ROOT / "notebooks", ROOT):
    text = str(entry)
    if text not in sys.path:
        sys.path.insert(0, text)

from helpers import accepted_pdfs

from models.query import QuestionSchema, QuestionType, UserQuestion

SHOW_TEXT = False
pdf_dir = (
    Path(os.environ["NOTEBOOK_PDF_DIR"])
    if os.environ.get("NOTEBOOK_PDF_DIR")
    else ROOT / "notebooks" / "inputs"
)
schema = QuestionSchema(
    name="Notebook schema",
    questions=[
        UserQuestion(
            label="Field",
            question="Copy the requested value from the document.",
            question_type=QuestionType.EXTRACTION,
        )
    ],
)
accepted, rejected = accepted_pdfs(pdf_dir)
print(f"accepted: {len(accepted)}  rejected: {len(rejected)}")
for path, reason in rejected:
    print(f"{path.name}\trejected: {reason}")

In [ ]:
import time

from helpers import new_session_id, timed

from pipeline.orchestrator import run

session_id = new_session_id()
events = []


def on_progress(event):
    event.stamp = time.perf_counter()
    events.append(event)


with timed("run") as run_time:
    results = await run(accepted, schema, session_id, on_progress)

print(session_id)
print(f"run: {run_time['seconds']:.3f}s")

In [ ]:
from helpers import answered_counts, kpi_summary, phase_durations

failures = [event for event in events if event.stage.value == "error"]
durations = phase_durations(events)
durations["run"] = run_time["seconds"]
print(durations)
print(kpi_summary(results, failures, durations))
for count in answered_counts(results):
    print(count)
for event in failures:
    print(event.source, type(event.error).__name__, event.error)

In [ ]:
from helpers import preview, result_rows

for row in result_rows(results):
    print(row)

if SHOW_TEXT:
    for document in results:
        for answer in document.answers:
            citation = answer.citation.text if answer.citation else None
            print(answer.document, answer.question.label)
            print(preview(answer.answer, SHOW_TEXT))
            print(preview(citation, SHOW_TEXT))

In [ ]:
from pipeline.embedder import delete_collection

delete_collection(session_id)
print(session_id, "deleted")